**till now we saw batch gradient descent**

- but this is memory inefficient
- slow convergence

**Solution**
- divide data into batches and train on all batches
- called as mini batch gradient descent

In [32]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [33]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
print(df.head())
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  smoothness

In [34]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

X_train_tensor.shape, y_train_tensor.shape

(torch.Size([455, 30]), torch.Size([455]))

In [5]:
# here we could use two for loops
#
'''

batch_size = 32

epochs = 25

n_samples = len(X_train_tensor)

for epoch in range(epochs):

    # Simply loop over the dataset in chunks of 'batch_size'

    for start_idx in range(0, n_samples, batch_size):

        end_idx = start_idx + batch_size

        X_batch = X_train_tensor[start_idx:end_idx]

        y_batch = y_train_tensor[start_idx:end_idx]

        # Forward pass

        y_pred = model(X_batch)

        loss = loss_function(y_pred, y_batch.view(-1, 1))

        # Update step

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

'''
print()

#### problem with this approach
1. no standard interface for data
2. no easy way to apply transformations
  - we want to do data augmentation
3. shuffling and sampling
4. batch management and parallelization.

##### solution
- dataset and dataloader

# Dataset and Dataloader classes

# PyTorch Dataset and DataLoader

## Overview

`Dataset` and `DataLoader` are two core abstractions in PyTorch that separate:

* **How data is stored and accessed** (`Dataset`)
* **How data is efficiently fed to the model** (`DataLoader`)

This separation makes the training pipeline flexible, scalable, and memory efficient.

---

# Dataset Class

The `Dataset` class acts as a blueprint that defines how data should be loaded and returned.

When creating a custom dataset, we decide:

* Where the data comes from
* How each sample is processed
* What is returned for a given index

A custom dataset must implement three methods:

## 1. `__init__()`

The constructor is responsible for loading and storing the dataset.

```python
def __init__(self):
    ...
```

Responsibilities:

* Read CSV files
* Load images
* Store labels
* Perform initial preprocessing

---

## 2. `__len__()`

Returns the total number of samples in the dataset.

```python
def __len__(self):
    ...
```

Example:

```python
return 1000
```

This tells PyTorch that the dataset contains 1000 samples.

---

## 3. `__getitem__(index)`

Returns the sample and label corresponding to a given index.

```python
def __getitem__(self, index):
    ...
```

Example:

```python
return X[index], y[index]
```

When index = 5:

```python
dataset[5]
```

returns the 5th data sample and its label.

---

# DataLoader Class

The `DataLoader` wraps a dataset and automates:

* Batching
* Shuffling
* Sampling
* Parallel data loading

Instead of manually creating batches, the DataLoader handles everything and feeds batches directly to the training loop.

---

# Example

Suppose we have a dataset containing **10 rows**.

```text
Indices:
0 1 2 3 4 5 6 7 8 9
```

and

```python
batch_size = 2
```

Then:

```text
Number of batches = 10 / 2 = 5
```

Possible batches:

```text
Batch 1 -> [0, 1]
Batch 2 -> [2, 3]
Batch 3 -> [4, 5]
Batch 4 -> [6, 7]
Batch 5 -> [8, 9]
```

---

# DataLoader Workflow

## Step 1: Shuffle Indices

At the beginning of each epoch, if

```python
shuffle=True
```

the DataLoader randomly shuffles all dataset indices.

Example:

```text
Original:
[0,1,2,3,4,5,6,7,8,9]

After Shuffling:
[7,3,1,8,5,0,2,9,4,6]
```

---

## Step 2: Create Batches of Indices

The shuffled indices are divided into chunks of size `batch_size`.

Example:

```text
[7,3]
[1,8]
[5,0]
[2,9]
[4,6]
```

These are only indices, not actual data samples.

---

## Step 3: Fetch Samples Using `__getitem__()`

For every index in the batch, DataLoader calls:

```python
dataset.__getitem__(index)
```

Example:

```text
dataset[7]
dataset[3]
```

which returns:

```text
(X7, y7)
(X3, y3)
```

---

## Step 4: Combine Samples into a Batch

The retrieved samples are passed to a function called:

```python
collate_fn
```

The default `collate_fn` stacks individual samples together.

Example:

```text
(X7, y7)
(X3, y3)
```

becomes

```text
X_batch =
[
    X7,
    X3
]

y_batch =
[
    y7,
    y3
]
```

---

## Step 5: Send Batch to Training Loop

The final batch is returned by the DataLoader and fed into the training pipeline.

```python
for X_batch, y_batch in dataloader:

    y_pred = model(X_batch)

    loss = loss_fn(y_pred, y_batch)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()
```

---

# Complete Data Flow

```text
Dataset
   │
   ▼
All Dataset Indices
[0,1,2,3,4,5,6,7,8,9]
   │
   ▼
Shuffle (Optional)
[7,3,1,8,5,0,2,9,4,6]
   │
   ▼
Create Batches of Indices
[7,3] [1,8] [5,0] [2,9] [4,6]
   │
   ▼
Call Dataset.__getitem__()
   │
   ▼
Retrieve Individual Samples
(X7,y7), (X3,y3)
   │
   ▼
collate_fn
   │
   ▼
Create Batch
(X_batch, y_batch)
   │
   ▼
Training Pipeline
Forward Pass
Loss Calculation
Backward Pass
Weight Update
```

---

# Key Difference

| Dataset                                         | DataLoader                                             |
| ----------------------------------------------- | ------------------------------------------------------ |
| Defines how data is loaded                      | Defines how data is batched and delivered              |
| Returns one sample at a time                    | Returns a batch of samples                             |
| Implements `__init__`, `__len__`, `__getitem__` | Handles batching, shuffling, sampling, multiprocessing |
| Focuses on data access                          | Focuses on efficient iteration                         |

### One-Line Summary

**Dataset tells PyTorch how to access a single sample, while DataLoader tells PyTorch how to efficiently deliver batches of samples to the training loop.**


In [7]:
from sklearn.datasets import make_classification
X, y = make_classification(
    n_samples = 10,
    n_features = 2,
    n_informative = 2,
    n_redundant = 0,
    n_classes = 2,
    random_state = 42
)

In [8]:
X.shape, y.shape

((10, 2), (10,))

In [9]:
X = torch.tensor(X, dtype = torch.float32)
y = torch.tensor(y, dtype = torch.long)

In [10]:
from torch.utils.data import Dataset, DataLoader

In [13]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self, index):

    return self.features[index], self.labels[index]

In [14]:
dataset = CustomDataset(X, y)

In [15]:
len(dataset)

10

In [18]:
dataset[3]

(tensor([-0.7206, -0.9606]), tensor(0))

In [19]:
dataloader = DataLoader(dataset, batch_size = 2, shuffle = True)

In [22]:
for batch_features, batch_labels in dataloader:
  print(batch_features)
  print(batch_labels)
  print("--"*20)


tensor([[-0.7206, -0.9606],
        [-0.9382, -0.5430]])
tensor([0, 1])
----------------------------------------
tensor([[-0.5872, -1.9717],
        [-1.1402, -0.8388]])
tensor([0, 0])
----------------------------------------
tensor([[ 1.0683, -0.9701],
        [-2.8954,  1.9769]])
tensor([1, 0])
----------------------------------------
tensor([[ 1.7774,  1.5116],
        [-1.9629, -0.9923]])
tensor([1, 0])
----------------------------------------
tensor([[ 1.7273, -1.1858],
        [ 1.8997,  0.8344]])
tensor([1, 1])
----------------------------------------


#### custom transformation
- like data augmentation
- on text data --> preprocessing, like loweercasing, lemmatization like that
- where?
- at in __getitem__ method since we return each row , so first we apply transformation then we will return it.

class CustomDataset(Dataset):

    def __init__(self, features, labels, transform=None):
        self.features = features
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):

        feature = self.features[index]
        label = self.labels[index]

        if self.transform:
            feature = self.transform(feature)

        return feature, label

#### Parallelization

- creating chunks of indices is sequential task then it is sent to getitem , item comes and then collate_fn creates batches

- we need to make it parallely using workers

#### Samplers

- Samplers control how data samples (indices) are drawn from the dataset.
- The `shuffle` parameter in DataLoader internally uses samplers.
- Samplers decide the order in which dataset indices are provided to the DataLoader.

Types of Samplers:

1. **Sequential Sampler**
   - Equivalent to `shuffle=False`
   - Indices are returned in order.
   - Example:

   ```text
   0, 1, 2, 3, 4, 5, 6, 7, 8, 9
   ```

   - Useful for:
     - Time Series Data
     - Sequential Data
     - Situations where order must be preserved

---

2. **Random Sampler**
   - Equivalent to `shuffle=True`
   - Indices are randomly shuffled before being returned.
   - Example:

   ```text
   7, 2, 9, 1, 5, 0, 8, 4, 3, 6
   ```

   - Useful for:
     - General Machine Learning tasks
     - Better model generalization
     - Preventing order bias

---

3. **Custom Sampler**
   - User-defined sampling strategy.
   - Gives full control over how samples are selected.
   - Useful for:
     - Imbalanced Datasets
     - Class-balanced sampling
     - Special sampling requirements

   Example:
   - Suppose Class A has very few samples.
   - We can design a sampler such that every batch contains at least one sample from Class A.

---

#### Summary

```text
Sampler
   │
   ├── SequentialSampler  → shuffle=False
   ├── RandomSampler      → shuffle=True
   └── CustomSampler      → User-defined sampling strategy
```

#### collate_fn

* `collate_fn` is a function used by the DataLoader to combine individual samples into a single batch.
* It defines how a list of samples returned by the Dataset should be merged.

Why do we need it?

* By default, DataLoader uses a simple collation mechanism that stacks samples into tensors.
* This works only when all samples have the same shape.
* In many real-world tasks, samples can have different sizes, so custom batching logic is required.

Common Use Case: NLP

* Text sequences often have variable lengths.

Example:

```text
Sentence 1 → [10, 5, 7]
Sentence 2 → [3, 8, 1, 9, 6]
```

* If `batch_size = 2`, these sequences cannot be stacked directly because their lengths are different.

Solution:

* Use a custom `collate_fn`.
* Apply padding to make all sequences in the batch the same length.

Example:

```text
Before Padding

[10, 5, 7]
[3, 8, 1, 9, 6]
```

```text
After Padding

[10, 5, 7, 0, 0]
[3, 8, 1, 9, 6]
```

* Now both sequences have the same length and can be combined into a batch.

Workflow:

```text
Dataset Samples
       │
       ▼
   collate_fn
       │
       ├── Padding
       ├── Custom Processing
       └── Batch Formation
       │
       ▼
     DataLoader
       │
       ▼
 Training Pipeline
```

Key Idea:

* `collate_fn` customizes how samples are combined into batches.
* It is especially useful when dealing with variable-length data such as text, audio, or sequences.


# DataLoader Parameters

### 1. `dataset`
- The Dataset object from which samples are loaded.
- DataLoader fetches data using the Dataset's `__getitem__()` method.

Example:

```python
DataLoader(dataset=my_dataset)
```

---

### 2. `batch_size`
- Number of samples in each batch.
- Determines how many samples are processed in one forward pass.

Example:

```python
batch_size = 32
```

---

### 3. `shuffle`
- Whether to shuffle the dataset before each epoch.
- Internally uses a Random Sampler.

Values:

```python
shuffle=True
```

- Random order of samples.

```python
shuffle=False
```

- Sequential order of samples.

---

### 4. `num_workers`
- Number of subprocesses used for loading data.
- Allows parallel data loading to improve training speed.

Example:

```python
num_workers = 4
```

- 4 worker processes load data in parallel.

---

### 5. `pin_memory`
- If `True`, DataLoader copies tensors into pinned (page-locked) memory.
- Speeds up CPU → GPU data transfer.

Typically used when training on GPU:

```python
pin_memory=True
```

---

### 6. `drop_last`
- Controls what happens when the last batch is smaller than `batch_size`.

Example:

```text
Dataset Size = 10
Batch Size = 3
```

Batches:

```text
[0,1,2]
[3,4,5]
[6,7,8]
[9]
```

```python
drop_last=False
```

- Keeps the last incomplete batch.

```python
drop_last=True
```

- Drops the last incomplete batch.

---

### 7. `collate_fn`
- Defines how individual samples are combined into a batch.
- Useful for custom batching logic.

Common use case:
- Variable-length text sequences
- Padding
- Custom preprocessing before batch creation

Example:

```python
collate_fn=my_collate_fn
```

---

### 8. `sampler`
- Controls how dataset indices are selected.
- Overrides the behavior of `shuffle`.

Types:
- SequentialSampler
- RandomSampler
- CustomSampler

Example:

```python
sampler=my_sampler
```

Useful for:
- Imbalanced datasets
- Class-balanced sampling
- Custom sampling strategies

---

## Summary

| Parameter | Purpose |
|------------|----------|
| `dataset` | Source of data |
| `batch_size` | Number of samples per batch |
| `shuffle` | Randomly shuffle data each epoch |
| `num_workers` | Parallel data loading |
| `pin_memory` | Faster CPU → GPU transfer |
| `drop_last` | Drop incomplete last batch |
| `collate_fn` | Custom batch formation logic |
| `sampler` | Controls how indices are sampled |

# our model

In [35]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return len(self.features)

  def __getitem__(self, idx):

    return self.features[idx], self.labels[idx]



In [36]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [37]:
train_dataset[10]

(tensor([-1.0074,  0.0028, -0.9867, -0.8855,  0.2786, -0.3565, -0.7283, -0.9292,
          1.4611,  0.2687, -0.1747,  0.5083, -0.2782, -0.3629,  0.1559, -0.2059,
         -0.1260, -0.5054,  0.7813, -0.1478, -0.9678, -0.3893, -0.9728, -0.8357,
         -0.4278, -0.6730, -0.9085, -1.2326, -0.1017, -0.4664]),
 tensor(0.))

In [38]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [39]:
import torch.nn as nn


class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):

    out = self.linear(features)
    out = self.sigmoid(out)

    return out

In [41]:
learning_rate = 0.1
epochs = 25

# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loss function
loss_function = nn.BCELoss()


# define loop
for epoch in range(epochs):

  for batch_features, batch_labels in train_loader:

    # forward pass
    # batch_features = batch_features.float()
    y_pred = model(batch_features)

    # loss calculate
    loss = loss_function(y_pred, batch_labels.view(-1,1))

    # clear gradients
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # parameters update
    optimizer.step()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.2811315357685089
Epoch: 2, Loss: 0.08484494686126709
Epoch: 3, Loss: 0.23085109889507294
Epoch: 4, Loss: 0.06135545298457146
Epoch: 5, Loss: 0.19547124207019806
Epoch: 6, Loss: 0.11863306909799576
Epoch: 7, Loss: 0.14263561367988586
Epoch: 8, Loss: 0.0575413815677166
Epoch: 9, Loss: 0.018629681318998337
Epoch: 10, Loss: 0.11464174836874008
Epoch: 11, Loss: 0.053114671260118484
Epoch: 12, Loss: 0.04621989652514458
Epoch: 13, Loss: 0.26228469610214233
Epoch: 14, Loss: 0.007635032292455435
Epoch: 15, Loss: 0.024339044466614723
Epoch: 16, Loss: 0.03674035519361496
Epoch: 17, Loss: 0.006435285788029432
Epoch: 18, Loss: 0.02904248796403408
Epoch: 19, Loss: 0.019595341756939888
Epoch: 20, Loss: 0.019688455387949944
Epoch: 21, Loss: 0.0690845400094986
Epoch: 22, Loss: 0.017134064808487892
Epoch: 23, Loss: 0.18566061556339264
Epoch: 24, Loss: 0.006862611509859562
Epoch: 25, Loss: 0.014785783365368843


In [42]:
# Model evaluation using test_loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')


Accuracy: 0.9531
